# Gesundheitskosten 2011-2026

Dieses Notebook lädt die Gesundheitskosten über die SDMX-API und beschränkt den Zeitraum auf 2011-2026.

In [ ]:
from pathlib import Path

import pandas as pd
import requests

In [ ]:
DATA_DIR = Path("../../data")
RAW_DIR = Path(DATA_DIR / "raw")
INTERIM_DIR = Path(DATA_DIR / "interim")

DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)
INTERIM_DIR.mkdir(exist_ok=True)

START_YEAR = 2011
END_YEAR = 2026

RAW_FILE = RAW_DIR / f"gesundheitskosten_raw_{START_YEAR}_{END_YEAR}.csv"
OUTPUT_FILE = INTERIM_DIR / f"gesundheitskosten_{START_YEAR}_{END_YEAR}.csv"

SDMX_KEY = "_T._T._T._T.._T..CHF.A"

BFS_COSTS_URL = (
    "https://disseminate.stats.swiss/rest/data/"
    "CH1.COU,DF_COU_HEALTH_COSTS,1.0.0/"
    f"{SDMX_KEY}"
    f"?startPeriod={START_YEAR}&endPeriod={END_YEAR}&dimensionAtObservation=AllDimensions"
)

HEADERS = {
    "Accept": "application/vnd.sdmx.data+csv;version=2.0;labels=both"
}

## Download

In [ ]:
def download_file(url: str, target: Path, headers: dict | None = None) -> None:
    if target.exists():
        return

    with requests.get(url, headers=headers, stream=True, timeout=300) as response:
        response.raise_for_status()

        with target.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

download_file(BFS_COSTS_URL, RAW_FILE, HEADERS)

## Code- und Label-Spalten trennen

Die Header Option `labels=both` gruppiert die Codes und Labels, diese müssen getrennt werden.

In [ ]:
COLUMN_SPLITS = {
    "AGE: Age groups": ("AGE", "Age groups"),
    "CANTON: Swiss cantons": ("CANTON", "Swiss cantons"),
    "TIME_PERIOD: Time Period": ("TIME_PERIOD", "Time Period"),
    "DIFF_REGION_STATE: Date of spatial reference": ("DIFF_REGION_STATE", "Date of spatial reference"),
    "DIFF_LAST_UPDATE: Update date": ("DIFF_LAST_UPDATE", "Update date"),
    "DIFF_EMBARGO_DATE: Date of publication": ("DIFF_EMBARGO_DATE", "Date of publication"),
    "DIFF_DB_STATE: Database state": ("DIFF_DB_STATE", "Database state"),
    "MULT: Unit Multiplier": ("MULT", "Unit Multiplier"),
    "OBS_STATUS: Code list for Observation Status": ("OBS_STATUS", "Code list for Observation Status"),
    "DECIMALS: Decimals": ("DECIMALS", "Decimals"),
    "DIFF_REGION_REF: Spatial reference": ("DIFF_REGION_REF", "Spatial reference"),
}

FIXED_DIMENSIONS = {
    "P": ("_T", "Total", "Service provider"),
    "S": ("_T", "Total", "Service"),
    "M": ("_T", "Total", "Mode of provision"),
    "F": ("_T", "Total", "Financing scheme"),
    "GENDER": ("_T", "Total", "Sex"),
    "UNIT": ("CHF", "Swiss franc", "Measurement unit"),
    "FREQ": ("A", "Annual", "Frequency"),
}

FINAL_COLUMNS = [
    "STRUCTURE", "STRUCTURE_ID", "STRUCTURE_NAME", "ACTION",
    "P", "Service provider", "S", "Service", "M", "Mode of provision",
    "F", "Financing scheme", "AGE", "Age groups", "GENDER", "Sex",
    "CANTON", "Swiss cantons", "UNIT", "Measurement unit", "FREQ", "Frequency",
    "TIME_PERIOD", "Time Period", "OBS_VALUE", "Observation value (DotStat)",
    "DIFF_REGION_STATE", "Date of spatial reference", "DIFF_LAST_UPDATE", "Update date",
    "DIFF_EMBARGO_DATE", "Date of publication", "DIFF_DB_STATE", "Database state",
    "MULT", "Unit Multiplier", "OBS_STATUS", "Code list for Observation Status",
    "DECIMALS", "Decimals", "DIFF_REGION_REF", "Spatial reference",
]

In [ ]:
def split_code_label(series: pd.Series) -> pd.DataFrame:
    parts = series.astype("string").str.split(": ", n=1, expand=True)

    if parts.shape[1] == 1:
        parts[1] = pd.NA

    return parts


def transform_bfs_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    result = pd.DataFrame(index=chunk.index)

    result["STRUCTURE"] = chunk["STRUCTURE"]
    result["STRUCTURE_ID"] = chunk["STRUCTURE_ID"]
    result["STRUCTURE_NAME"] = "Costs of health care"
    result["ACTION"] = chunk["ACTION"]

    for code_col, (code_value, label_value, label_col) in FIXED_DIMENSIONS.items():
        result[code_col] = code_value
        result[label_col] = label_value

    for source_col, (code_col, label_col) in COLUMN_SPLITS.items():
        split_values = split_code_label(chunk[source_col])
        result[code_col] = split_values[0]
        result[label_col] = split_values[1]

    result["OBS_VALUE"] = chunk["OBS_VALUE"]
    result["Observation value (DotStat)"] = pd.NA

    return result[FINAL_COLUMNS]

In [ ]:
first_chunk = True

for chunk in pd.read_csv(RAW_FILE, chunksize=250_000):
    transformed = transform_bfs_chunk(chunk)

    transformed.to_csv(
        OUTPUT_FILE,
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk,
    )

    first_chunk = False

## Validierung

In [ ]:
df_check = pd.read_csv(OUTPUT_FILE)
df_check.head()